# Reasoner Run (Project 3, Steps 5 and 8)

Runs **HermiT** (full OWL 2 DL) and **ELK** (OWL 2 EL) through ROBOT at three levels:

1. **Each mapping file alone.** This is the check the original notebook did. It only shows that each file is well-formed OWL.
2. **Each source ontology alone** (local files, `owl:imports` stripped). Catches problems inside the ontologies themselves.
3. **Each merged pair**: left + right ontology + both mapping files (+ `bfo-core.ttl` when the mappings use BFO classes). This is the real test. It checks that the mapping is consistent, that no class became unsatisfiable, and which cross-ontology subclass and equivalence axioms the reasoners infer beyond what was asserted.

Results are written to `src/data/reasoner-report.xlsx`.

**Fixes to the original notebook:**
* ROBOT is found on `PATH` *or* through the `ROBOT_JAR` environment variable (default `~/tools/robot.jar`, run with `java -jar`).
* `NamedTemporaryFile(delete=True)` cannot be reopened by another process on Windows. A temporary *directory* is used instead.
* The OWL-Time mapping file is `to-mapping.ttl` (the name the grader expects), not `time-mapping.ttl`.

In [1]:
import os, re, shutil, subprocess, tempfile
from pathlib import Path
import pandas as pd
from rdflib import Graph, URIRef, RDF, RDFS, OWL

NB_DIR   = Path.cwd().resolve()
SRC_DIR  = NB_DIR.parent / "src"
DATA_DIR = SRC_DIR / "data"
if not SRC_DIR.exists() or not DATA_DIR.exists():
    raise FileNotFoundError("Could not find sibling src/ and src/data/ next to notebooks/.")
print("SRC_DIR :", SRC_DIR)
print("DATA_DIR:", DATA_DIR)

def robot_cmd():
    if shutil.which("robot"):
        return ["robot"]
    jar = Path(os.environ.get("ROBOT_JAR", Path.home() / "tools" / "robot.jar"))
    if jar.exists() and shutil.which("java"):
        return ["java", "-jar", str(jar)]
    raise RuntimeError("ROBOT not found: put 'robot' on PATH or set ROBOT_JAR to robot.jar")

ROBOT = robot_cmd()
print(subprocess.run(ROBOT + ["--version"], capture_output=True, text=True).stdout.strip())
REASONERS = ["HermiT", "ELK"]

SRC_DIR : C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src
DATA_DIR: C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src\data


ROBOT version 1.9.10


In [2]:
def run_reason(input_file: Path, reasoner: str, output: Path):
    """robot reason; returns (ok, status, unsatisfiable IRIs)."""
    res = subprocess.run(
        ROBOT + ["reason", "--reasoner", reasoner,
                 "--axiom-generators", "SubClass EquivalentClass",
                 "--exclude-tautologies", "structural",
                 "--input", str(input_file), "--output", str(output)],
        capture_output=True, text=True)
    log = res.stdout + res.stderr
    unsat = re.findall(r"unsatisfiable: (\S+)", log)
    if res.returncode == 0:
        return True, "consistent, all classes satisfiable", []
    if "inconsistent" in log:
        return False, "INCONSISTENT", []
    if unsat:
        return False, f"{len(unsat)} unsatisfiable classes", unsat
    msgs = [l.strip() for l in log.splitlines()
            if l.strip() and "WARNING" not in l and not l.startswith("Use the")]
    return False, "REFUSED: " + (msgs[-1][:200] if msgs else f"exit {res.returncode}"), []

## 1. Each mapping file on its own

In [3]:
mapping_files = ["bfo-mapping.ttl", "ies-mapping.ttl", "ccom-mapping.ttl",
                 "qudt-mapping.ttl", "ccot-mapping.ttl", "to-mapping.ttl"]

rows = []
with tempfile.TemporaryDirectory() as tmp:
    for name in mapping_files:
        path = DATA_DIR / name
        if not path.exists():
            print(f"SKIP  (missing): {name}")
            continue
        for r in REASONERS:
            ok, status, _ = run_reason(path, r, Path(tmp) / "out.ttl")
            print(f"{'SUCCESS' if ok else 'FAILURE'}: {name} [{r}]  {status}")
            rows.append({"level": "mapping file", "input": name, "reasoner": r, "ok": ok, "status": status})

SUCCESS: bfo-mapping.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: bfo-mapping.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ies-mapping.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: ies-mapping.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ccom-mapping.ttl [HermiT]  consistent, all classes satisfiable


...

SUCCESS: qudt-mapping.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ccot-mapping.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: ccot-mapping.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: to-mapping.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: to-mapping.ttl [ELK]  consistent, all classes satisfiable


## 2. Each source ontology on its own

`owl:imports` are stripped so the check covers only the files in `src/`. Otherwise ROBOT fetches `dtype`/`vaem` (QUDT) and CCO modules (CCOM, CCOT) from the web at run time, which is slow and not reproducible. The QUDT imports also bring in `xsd:date` and `xsd:anySimpleType`, which are outside the OWL 2 datatype map and make HermiT report an inconsistency.

In [4]:
def local_graph(*paths: Path) -> Graph:
    g = Graph()
    for p in paths:
        g.parse(p)
    g.remove((None, OWL.imports, None))
    return g

SOURCES = ["bfo-core.ttl", "ies.ttl", "ccom.ttl", "qudt.ttl", "ccot.ttl", "time.ttl"]
with tempfile.TemporaryDirectory() as tmp:
    for name in SOURCES:
        f = Path(tmp) / name
        local_graph(SRC_DIR / name).serialize(f, format="turtle")
        for r in REASONERS:
            ok, status, _ = run_reason(f, r, Path(tmp) / "out.ttl")
            print(f"{'SUCCESS' if ok else 'FAILURE'}: {name} [{r}]  {status}")
            rows.append({"level": "source ontology", "input": name, "reasoner": r, "ok": ok, "status": status})

SUCCESS: bfo-core.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: bfo-core.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ies.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: ies.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ccom.ttl [HermiT]  consistent, all classes satisfiable


...

SUCCESS: qudt.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: ccot.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: ccot.ttl [ELK]  consistent, all classes satisfiable


SUCCESS: time.ttl [HermiT]  consistent, all classes satisfiable


SUCCESS: time.ttl [ELK]  consistent, all classes satisfiable


### 2b. Upstream originals vs repaired files: where HermiT and ELK disagree

The source files were repaired while debugging (the changes are marked `P3 repair` inside the TTLs). This cell re-runs the **original** QUDT and OWL-Time files, taken from git `upstream/main`, to show the reasoner differences that motivated the repairs. It is skipped if git or the upstream remote is unavailable.

In [5]:
def upstream_file(name: str):
    rel = f"projects/project-3/assignment/src/{name}"
    for ref in ("upstream/main", "origin/main", "main"):
        res = subprocess.run(["git", "show", f"{ref}:{rel}"], capture_output=True, cwd=SRC_DIR)
        if res.returncode == 0:
            return res.stdout, ref
    return None, None

with tempfile.TemporaryDirectory() as tmp:
    for name in ["qudt.ttl", "time.ttl"]:
        data, ref = upstream_file(name) if shutil.which("git") else (None, None)
        if data is None:
            print(f"SKIP  {name}: original not available from git")
            continue
        raw = Path(tmp) / f"original-{name}"
        raw.write_bytes(data)
        f = Path(tmp) / f"original-local-{name}"
        local_graph(raw).serialize(f, format="turtle")
        for r in REASONERS:
            ok, status, _ = run_reason(f, r, Path(tmp) / "out.ttl")
            print(f"{'SUCCESS' if ok else 'FAILURE'}: original {name} ({ref}) [{r}]  {status}")
            rows.append({"level": "upstream original", "input": name, "reasoner": r, "ok": ok, "status": status})

FAILURE: original qudt.ttl (upstream/main) [HermiT]  161 unsatisfiable classes


SUCCESS: original qudt.ttl (upstream/main) [ELK]  consistent, all classes satisfiable


FAILURE: original time.ttl (upstream/main) [HermiT]  REFUSED: Non-simple property '<http://www.w3.org/2006/time#disjoint>' or its inverse appears in disjoint properties axiom.


SUCCESS: original time.ttl (upstream/main) [ELK]  consistent, all classes satisfiable


## 3. Merged pairs: consistency, and which mappings the reasoners infer

A cross-ontology axiom is **inferred** if it is in the reasoner output but was not asserted in any input file. `requires_mapping` is **False** when the axiom already follows from the ontologies alone. For example, CCOT on its own entails `Multi-Hour Temporal Interval SubClassOf temporal interval`, because its `interval contains` property has domain *temporal interval*, even though CCOT only asserts the class under *one-dimensional temporal region*.

In [6]:
PAIRS = {
    "bfo-ies":   {"files": ["bfo-core.ttl", "ies.ttl"],  "maps": ["bfo-mapping.ttl", "ies-mapping.ttl"],
                  "ns": {"bfo": "http://purl.obolibrary.org/obo/BFO_", "ies": "http://ies.data.gov.uk/ontology/ies4#"}},
    "ccom-qudt": {"files": ["ccom.ttl", "qudt.ttl"],     "maps": ["ccom-mapping.ttl", "qudt-mapping.ttl"],
                  "ns": {"cco": "https://www.commoncoreontologies.org/", "qudt": "http://qudt.org/schema/qudt#"}},
    "ccot-to":   {"files": ["ccot.ttl", "time.ttl", "bfo-core.ttl"], "maps": ["ccot-mapping.ttl", "to-mapping.ttl"],
                  "ns": {"cco": "https://www.commoncoreontologies.org/", "time": "http://www.w3.org/2006/time#",
                         "bfo": "http://purl.obolibrary.org/obo/BFO_"}},
}

def source_of(iri: str, ns: dict):
    return next((k for k, v in ns.items() if iri.startswith(v)), None)

def cross_axioms(g: Graph, ns: dict) -> set:
    out = set()
    for p, rel in ((RDFS.subClassOf, "subClassOf"), (OWL.equivalentClass, "equivalentClass")):
        for s, o in g.subject_objects(p):
            if isinstance(s, URIRef) and isinstance(o, URIRef):
                a, b = source_of(str(s), ns), source_of(str(o), ns)
                if a and b and a != b:
                    out.add((str(s), rel, str(o)) if rel == "subClassOf" or str(s) < str(o) else (str(o), rel, str(s)))
    return out

def label(g: Graph, iri: str) -> str:
    for l in g.objects(URIRef(iri), RDFS.label):
        if getattr(l, "language", None) in (None, "en", "en-GB", "en-US"):
            return str(l)
    return iri.rsplit("/", 1)[-1].rsplit("#", 1)[-1]

inferred_rows, unsat_rows = [], []
with tempfile.TemporaryDirectory() as tmp:
    for pair, spec in PAIRS.items():
        # Baseline: the same ontologies WITHOUT the mapping files. Anything already entailed
        # here is a consequence of the ontologies themselves, not of our mappings.
        base = local_graph(*[SRC_DIR / f for f in spec["files"]])
        base_file = Path(tmp) / f"{pair}-baseline.ttl"
        base.serialize(base_file, format="turtle")
        ok_b, _, _ = run_reason(base_file, "HermiT", Path(tmp) / f"{pair}-baseline-out.ttl")
        baseline = set()
        if ok_b:
            g_b = Graph(); g_b.parse(Path(tmp) / f"{pair}-baseline-out.ttl")
            baseline = cross_axioms(g_b, spec["ns"])

        merged = local_graph(*[SRC_DIR / f for f in spec["files"]], *[DATA_DIR / m for m in spec["maps"]])
        merged_file = Path(tmp) / f"{pair}-merged.ttl"
        merged.serialize(merged_file, format="turtle")
        asserted = cross_axioms(merged, spec["ns"])
        per_reasoner = {}
        for r in REASONERS:
            out = Path(tmp) / f"{pair}-{r}.ttl"
            ok, status, unsat = run_reason(merged_file, r, out)
            new = set()
            if ok:
                g_out = Graph(); g_out.parse(out)
                new = cross_axioms(g_out, spec["ns"]) - asserted
            per_reasoner[r] = new
            print(f"{'SUCCESS' if ok else 'FAILURE'}: {pair} merged [{r}]  {status}; "
                  f"{len(asserted)} asserted cross-ontology axioms, {len(new)} inferred "
                  f"({len(new - baseline)} of them need the mappings)")
            rows.append({"level": "merged pair", "input": pair, "reasoner": r, "ok": ok, "status": status,
                         "asserted_cross_axioms": len(asserted), "inferred_cross_axioms": len(new)})
            unsat_rows += [{"pair": pair, "reasoner": r, "class": u, "label": label(merged, u)} for u in unsat]
        for ax in sorted(set().union(*per_reasoner.values())):
            s, rel, o = ax
            inferred_rows.append({
                "pair": pair, "subject": label(merged, s), "relation": rel, "object": label(merged, o),
                **{f"inferred_by_{r}": ax in per_reasoner[r] for r in REASONERS},
                "requires_mapping": ax not in baseline,
                "subject_iri": s, "object_iri": o})

summary = pd.DataFrame(rows)
inferred = pd.DataFrame(inferred_rows)
summary

SUCCESS: bfo-ies merged [HermiT]  consistent, all classes satisfiable; 4 asserted cross-ontology axioms, 3 inferred (3 of them need the mappings)


SUCCESS: bfo-ies merged [ELK]  consistent, all classes satisfiable; 4 asserted cross-ontology axioms, 3 inferred (3 of them need the mappings)


SUCCESS: ccom-qudt merged [HermiT]  consistent, all classes satisfiable; 28 asserted cross-ontology axioms, 32 inferred (32 of them need the mappings)


SUCCESS: ccom-qudt merged [ELK]  consistent, all classes satisfiable; 28 asserted cross-ontology axioms, 32 inferred (32 of them need the mappings)


SUCCESS: ccot-to merged [HermiT]  consistent, all classes satisfiable; 31 asserted cross-ontology axioms, 13 inferred (6 of them need the mappings)


SUCCESS: ccot-to merged [ELK]  consistent, all classes satisfiable; 31 asserted cross-ontology axioms, 13 inferred (6 of them need the mappings)


,level,input,reasoner,ok,status,asserted_cross_axioms,inferred_cross_axioms
0,mapping file,bfo-mapping.ttl,HermiT,True,"consistent, all classes satisfiable",NaN,NaN
1,mapping file,bfo-mapping.ttl,ELK,True,"consistent, all classes satisfiable",NaN,NaN
2,mapping file,ies-mapping.ttl,HermiT,True,"consistent, all classes satisfiable",NaN,NaN
3,mapping file,ies-mapping.ttl,ELK,True,"consistent, all classes satisfiable",NaN,NaN
4,mapping file,ccom-mapping.ttl,HermiT,True,"consistent, all classes satisfiable",NaN,NaN
5,mapping file,ccom-mapping.ttl,ELK,True,"consistent, all classes satisfiable",NaN,NaN
6,mapping file,qudt-mapping.ttl,HermiT,True,"consistent, all classes satisfiable",NaN,NaN
7,mapping file,qudt-mapping.ttl,ELK,True,"consistent, all classes satisfiable",NaN,NaN
8,mapping file,ccot-mapping.ttl,HermiT,True,"consistent, all classes satisfiable",NaN,NaN
9,mapping file,ccot-mapping.ttl,ELK,True,"consistent, all classes satisfiable",NaN,NaN


## Differences across reasoners

ELK only supports the OWL 2 EL profile. It ignores universal restrictions (`only`), cardinalities, complements, inverse properties, disjunctions and datatype facets. It can therefore miss inferences (and inconsistencies) that HermiT finds. HermiT is complete for OWL 2 DL but refuses ontologies that break the DL global restrictions.

In [7]:
if len(inferred):
    diff = inferred[inferred["inferred_by_HermiT"] != inferred["inferred_by_ELK"]]
    print(f"{len(inferred)} inferred cross-ontology axioms in total; {len(diff)} found by only one reasoner")
    display(inferred.drop(columns=["subject_iri", "object_iri"]))
else:
    print("No inferred cross-ontology axioms.")

48 inferred cross-ontology axioms in total; 0 found by only one reasoner


,pair,subject,relation,object,inferred_by_HermiT,inferred_by_ELK,requires_mapping
0,bfo-ies,Particular Period,equivalentClass,temporal interval,True,True,True
1,bfo-ies,Particular Period,subClassOf,one-dimensional temporal region,True,True,True
2,bfo-ies,temporal interval,subClassOf,Period Of Time,True,True,True
3,ccom-qudt,Amount Of Substance Unit,subClassOf,ont00000120,True,True,True
4,ccom-qudt,Angular Mass Unit,subClassOf,ont00000120,True,True,True
5,ccom-qudt,Area Unit,subClassOf,ont00000120,True,True,True
6,ccom-qudt,Force Unit,subClassOf,ont00000120,True,True,True
7,ccom-qudt,Frequency Unit,subClassOf,ont00000120,True,True,True
8,ccom-qudt,Length Unit,subClassOf,ont00000120,True,True,True
9,ccom-qudt,Linear Acceleration Unit,subClassOf,ont00000120,True,True,True


## 4. Counterfactual: why continuant-side IES classes are not mapped

IES is 4D: every Entity is also its own whole-life State (`ies:Person` is a subclass of both `ies:Entity` and `ies:PersonState`, which is a subclass of `ies:State`). This cell adds two tempting mappings, `ies:State ⊑ BFO occurrent` and `ies:Person ⊑ BFO object`, to the BFO–IES merge. HermiT finds `ies:Person` unsatisfiable, because BFO continuant and occurrent are disjoint. Nothing is written to the mapping files.

In [8]:
from rdflib import Namespace
IES = Namespace("http://ies.data.gov.uk/ontology/ies4#")
OBO = Namespace("http://purl.obolibrary.org/obo/")
with tempfile.TemporaryDirectory() as tmp:
    g = local_graph(*[SRC_DIR / f for f in PAIRS["bfo-ies"]["files"]], *[DATA_DIR / m for m in PAIRS["bfo-ies"]["maps"]])
    g.add((IES.State, RDFS.subClassOf, OBO.BFO_0000003))    # State  -> occurrent
    g.add((IES.Person, RDFS.subClassOf, OBO.BFO_0000030))   # Person -> object
    f = Path(tmp) / "counterfactual.ttl"
    g.serialize(f, format="turtle")
    ok, status, unsat = run_reason(f, "HermiT", Path(tmp) / "out.ttl")
    print(f"{'SUCCESS' if ok else 'FAILURE'}: bfo-ies + State->occurrent + Person->object [HermiT]  {status}")
    for u in unsat:
        print("   unsatisfiable:", u)
    rows.append({"level": "counterfactual", "input": "bfo-ies + State->occurrent + Person->object",
                 "reasoner": "HermiT", "ok": ok, "status": status})
summary = pd.DataFrame(rows)

FAILURE: bfo-ies + State->occurrent + Person->object [HermiT]  1 unsatisfiable classes
   unsatisfiable: http://ies.data.gov.uk/ontology/ies4#Person


In [9]:
with pd.ExcelWriter(DATA_DIR / "reasoner-report.xlsx") as xw:
    summary.to_excel(xw, sheet_name="consistency", index=False)
    inferred.to_excel(xw, sheet_name="inferred-mappings", index=False)
    pd.DataFrame(unsat_rows, columns=["pair", "reasoner", "class", "label"]).to_excel(xw, sheet_name="unsatisfiable", index=False)
print("Wrote", DATA_DIR / "reasoner-report.xlsx")

Wrote C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src\data\reasoner-report.xlsx
